# 14. Külső POI Adatintegráció és "15 perces város" Elemzés

**Cél**: Kőbánya "15-perces város" jellegének mérése OpenStreetMap (OSM) Point of Interest (POI) adatok integrációjával.
A modell megvizsgálja, hogy a zöldfelületek, oktatási és vendéglátóipari szolgáltatások közelsége hogyan épül be a lakásárakba.

**Módszertan**: Az `osmnx` csomag segítségével élő térképi adatokat (amenities, parks) kérünk le Kőbányára, majd ezeket összekötjük az ingatlanok koordinátáival.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
try:
    import osmnx as ox
    import geopandas as gpd
    from shapely.geometry import Point
    OSMNX_AVAILABLE = True
except ImportError:
    OSMNX_AVAILABLE = False
    print("Figyelem: az 'osmnx' és 'geopandas' csomagok telepítése javasolt a teljes funkcióhoz.")

df = load_szamitott_master()
df_pontos = df[(df['minta_garantalt_pontos'] == 1) & (df['listing_type'] == 'elado')].copy()
print(f"Elemzett minta (garantált pontos): {len(df_pontos)} db.")

### 1. POI Adatok Lekérése (OSM) és Kőbánya Hálózata
OSM adatok betöltése Kőbánya közigazgatási határán belül (vagy betöltése cache-ből).

In [ ]:
if OSMNX_AVAILABLE:
    # Egyszerűsített bounding box Kőbányára
    place_name = 'Kőbánya, Budapest, Hungary'
    
    try:
        # Próbáljuk meg lekérni a parkokat és éttermeket
        tags = {'leisure': 'park', 'amenity': ['restaurant', 'cafe', 'school']}
        poi_data = ox.features_from_place(place_name, tags)
        print(f"Sikeresen lekérve {len(poi_data)} db POI adat az OpenStreetMap-ről.")
        
        # Geometriai középpontok (centroidok) számítása
        poi_data = poi_data.to_crs(epsg=3857) # Vetület a távolsághoz
        poi_data['centroid'] = poi_data.geometry.centroid
        poi_data = poi_data.to_crs(epsg=4326) # Vissza WGS84-be
        
    except Exception as e:
        print(f"Nem sikerült élőben lekérni az OSM adatokat. Hiba: {e}")
        poi_data = None
else:
    print("OSMNX hiányzik, az elemzés csak demó módban fut.")
    poi_data = None

### 2. A "15-Perces Város" Index Kiszámítása
A hirdetések koordinátái alapján megnézzük, hány szolgáltatás (POI) érhető el az 5-10-15 perces gyalogos sávokban.
Módszertanilag szinkronizáljuk a távolságokat az NB05-ös izokrón sávokkal (5 perc = 375m, 10 perc = 750m, 15 perc = 1125m).

In [ ]:
# POI indexek számítása a standard gyalogos sávokra (légvonalbeli közelítés)
if OSMNX_AVAILABLE and poi_data is not None:
    # Ingatlanok geometriája
    gdf_ing = gpd.GeoDataFrame(
        df_pontos, 
        geometry=gpd.points_from_xy(df_pontos.geokodolt_lon, df_pontos.geokodolt_lat),
        crs="EPSG:4326"
    )
    
    # 3857 vetületre a méter alapú távolságokhoz
    gdf_ing_3857 = gdf_ing.to_crs(epsg=3857)
    poi_3857 = poi_data.to_crs(epsg=3857)
    
    # Szolgáltatások száma a standard sávokban
    poi_375 = []
    poi_750 = []
    poi_1125 = []
    
    for idx, row in gdf_ing_3857.iterrows():
        point = row.geometry
        distances = poi_3857.geometry.distance(point)
        poi_375.append((distances <= 375).sum())
        poi_750.append((distances <= 750).sum())
        poi_1125.append((distances <= 1125).sum())
        
    df_pontos['poi_375m_count'] = poi_375
    df_pontos['poi_750m_count'] = poi_750
    df_pontos['poi_1125m_count'] = poi_1125
    
    fig = px.scatter_map(
        df_pontos, lat='geokodolt_lat', lon='geokodolt_lon', color='poi_1125m_count',
        size='nm_ar_huf', hover_name='cim_teljes', map_style='carto-positron',
        title='"15 perces város" - Szolgáltatások száma 1125 méteren belül'
    )
    fig.update_layout(height=500, margin={"r":0,"t":40,"l":0,"b":0})
    fig.show()
else:
    print("POI adatok hiányában a számítás szimulált adatokat mutat (placeholder).")
    df_pontos['poi_375m_count'] = np.random.randint(1, 10, len(df_pontos))
    df_pontos['poi_750m_count'] = np.random.randint(5, 20, len(df_pontos))
    df_pontos['poi_1125m_count'] = np.random.randint(10, 50, len(df_pontos))
    
    kpi_cards = [
        ("Átlagos POI 1125m-en belül", f"{df_pontos['poi_1125m_count'].mean():.1f} db", "Szimulált adat", "#1e3a8a")
    ]
    display(HTML(kpi_grid_html(kpi_cards)))

### 3. Hedonikus Árprémium a POI Sűrűség alapján
Az OSM szolgáltatási sűrűség (3 gyalogos sávban mérve) beépítése a lakásárak lineáris regressziójába.

In [ ]:
import statsmodels.api as sm

features = ['korrigalt_alapterulet_nm', 'is_panel', 'tavolsag_metro_halozati_m', 'poi_375m_count', 'poi_750m_count', 'poi_1125m_count']
df_reg = df_pontos.dropna(subset=['log_nm_ar'] + features).copy()

X = sm.add_constant(df_reg[features])
y = df_reg['log_nm_ar']
model = sm.OLS(y, X).fit()

res_df = pd.DataFrame({
    'Változó': model.params.index,
    'Együttható (β)': model.params.values,
    'p-érték': model.pvalues.values
})
display(HTML("<div style='overflow-x:auto; margin: 15px 0;'>" + res_df.round(4).to_html(classes='table table-bordered table-striped', index=False) + "</div>"))
print(f"Az új modell R² értéke: {model.rsquared:.4f}")